# **Data Visualization and Analysis**


In this section, we perform a comprehensive analysis of the paleobiological dataset focusing on the Dinosauria and Pterosauria taxons. The analysis aims to uncover patterns, trends, and insights regarding the diversity, distribution, and evolutionary history of these ancient species.

***Geographical Analysis***: Using the coordinates of fossil findings, we map the geographical spread of species, revealing hotspots of paleobiological activity and potential correlations with ancient environmental conditions.

***Species Distribution***: We analyze the distribution of species across different taxonomic orders, periods, and geographical locations. Visualizations such as bar charts, pie charts, and maps are employed to highlight diversity and prevalence within these groups.

- **Analyze Basal Species for orders, superorders, and infraorders (look at max MYA)**
- **Analyze Genus, Order, and Fossil Occ Distribution**
- **etc.**

***Temporal Analysis***: By examining the first and last appearances of species, we create timelines to track the evolutionary lifespan and extinction patterns. This helps to identify periods of significant biodiversity or extinction events.

***Lifespan Analysis***: The analysis of species lifespans across different orders and periods provides insights into the survival and adaptation strategies of these taxa. Statistical comparisons and visual summaries are used to present the findings.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib as mpl
import folium as fm
import requests as rq
import io

The first step we need to do is pull the data from the Collection. We can do this by either running that notebook from the current one or by using the CSV file.

In [2]:
# Importing the dataframes from the Data Collection Notebook to use in this one
#%run dino_collection.ipynb

# Importing the data from the saved CSV files from the previous notebook
species = pd.read_csv('../data-reserve/species-list.csv')
species = species.drop(columns=['Formation', 'Coordinates'])
genera = pd.read_csv('../data-reserve/genera-list.csv')
species.head()

,Species,Genus,Family,Infraorder,Suborder,Order,Informal,Diet,Early Age,Late Age,Early Period,Late Period,Max MYA,Min MYA,Lifespan (MYA)
0,Ajkaceratops kozmai,Ajkaceratops,NaN,NaN,NaN,Ornithischia,False,Herbivore,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,86.3,83.6,2.7
1,Turanoceratops tardabilis,Turanoceratops,NaN,NaN,NaN,Ornithischia,False,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1
2,Zuniceratops christopheri,Zuniceratops,NaN,NaN,NaN,Ornithischia,False,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1
3,Bagaceratops rozhdestvenskyi,Bagaceratops,Protoceratopsidae,NaN,NaN,Ornithischia,False,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5
4,Breviceratops kozlowskii,Breviceratops,Protoceratopsidae,NaN,NaN,Ornithischia,False,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5


Now that we have imported the dataframes, we need can continue with our analysis. 

**As of 2/2/2025, the ***Taxonomy of Fossil Occurrences**** URL does not pull any records. As the team at the US National Science Foundation and UW-Madison Dept of Geoscience look into the issue, we will need to add a few more steps in our code. However, please note that the following code block should only be used in cases like this, when PaleoDB is not pulling the correct records.* We need to grab the Occurrences data once more so we can use that to plot the map. Once that is complete, we can re-add the coordinate and formation fields as we did previously.

In [9]:
# Scraping data from the Paleobiology Database and cleaning up the Null values
occ_url = rq.get('https://paleobiodb.org/data1.2/occs/list.csv?base_name=Dinosauria&taxon_reso=species&idqual=certain&pres=regular&max_ma=252&min_ma=65&show=class,coords,loc,strat,acconly').content
occ = pd.read_csv(io.StringIO(occ_url.decode('utf-8')))[['accepted_name', 'lng', 'lat', 'formation']]

occ.columns = ['Species', 'Longitude', 'Latitude', 'Formation']
occ['Coordinates'] = tuple(zip(occ['Latitude'], occ['Longitude']))

occ = occ.groupby(occ['Species']).aggregate({'Formation' : 'unique', 'Coordinates' : 'unique'}).reset_index()

species = species.merge(occ, on='Species', how='left').fillna('Not Applicable')
species.head()

,Species,Genus,Family,Infraorder,Suborder,Order,Informal,Diet,Early Age,Late Age,Early Period,Late Period,Max MYA,Min MYA,Lifespan (MYA),Formation,Coordinates
0,Ajkaceratops kozmai,Ajkaceratops,Not Applicable,Not Applicable,Not Applicable,Ornithischia,False,Herbivore,Santonian,Santonian,Upper Cretaceous,Upper Cretaceous,86.3,83.6,2.7,[Csehbánya],"[(47.216702, 17.6)]"
1,Turanoceratops tardabilis,Turanoceratops,Not Applicable,Not Applicable,Not Applicable,Ornithischia,False,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1,[Bissekty],"[(42.117294, 62.655315)]"
2,Zuniceratops christopheri,Zuniceratops,Not Applicable,Not Applicable,Not Applicable,Ornithischia,False,Herbivore,Turonian,Turonian,Upper Cretaceous,Upper Cretaceous,93.9,89.8,4.1,[Moreno Hill],"[(35.066666, -108.849998)]"
3,Bagaceratops rozhdestvenskyi,Bagaceratops,Protoceratopsidae,Not Applicable,Not Applicable,Ornithischia,False,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5,"[Baruungoyot, Bayan Mandahu]","[(43.25, 99.75), (43.299999, 99.599998), (41.7..."
4,Breviceratops kozlowskii,Breviceratops,Protoceratopsidae,Not Applicable,Not Applicable,Ornithischia,False,Herbivore,Campanian,Campanian,Upper Cretaceous,Upper Cretaceous,83.6,72.1,11.5,[Baruungoyot],"[(43.487499, 101.125)]"


## **Geographical Analysis**

In [16]:
map = fm.Map(location=[20, 0], 
             zoom_start=1.75, 
             max_bounds=True,
             min_lat=-90,
             max_lat=90,
             min_lon=-180,
             max_lon=180)

marker_colors = ['red', 'green', 'blue']

for index, i in species.iterrows():
    
    for coords in i['Coordinates']:
        
        perd = str(i['Early Period']) 
        if (i['Early Period'] != i['Late Period']):
            perd += ' - ' + str(i['Late Period'])
        
        diet = i['Diet']
        mc = marker_colors[0] if diet == 'Carnivore' else (marker_colors[1] if diet == 'Herbivore' else marker_colors[2])
            
        pops = fm.IFrame('<h3><b><i>' + i['Species'] + '</i></b></h3><br><b>Diet:</b> ' + diet + '<br><b>Period:</b> ' + perd)
        
        
        fm.CircleMarker(location=[coords[0], coords[1]],
                        radius=5, popup=fm.Popup(pops, min_width = 250, max_width = 250),
                        color=mc).add_to(map)
        
display(map)